### Imports & Setup

In [ ]:
%pip install uv --quiet
%uv pip install pandas numpy scipy plotly requests

In [15]:
# Data manipulation tools
import io
import pandas as pd
import numpy as np
from scipy import stats

# Visualization tools
import plotly.express as px
import plotly.graph_objects as go

# Network tools
import requests

# OS tools
from pathlib import Path
import sys

repo_root = Path.cwd().parent

### Load Data

In [16]:
df = pd.read_csv(repo_root / "all_stocks.csv", parse_dates=["Date"])

df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Company
0,2021-01-01,13.305857,13.613475,12.550024,13.066792,1925304000,NVDA,NVIDIA Corporation
1,2021-01-08,13.162520,13.921097,13.004723,13.324307,1595496000,NVDA,NVIDIA Corporation
2,2021-01-15,13.827864,13.959237,12.807289,13.190191,1100320000,NVDA,NVIDIA Corporation
3,2021-01-22,13.013695,13.802185,12.757180,13.700725,1261500000,NVDA,NVIDIA Corporation
4,2021-01-29,13.625196,13.891931,12.865872,13.037629,1155944000,NVDA,NVIDIA Corporation


### Stock Closing Prices vs. Inflation

In [17]:
INFLATION_CACHE = repo_root / "data" / "cpi_cache.csv"

def get_inflation():
    if INFLATION_CACHE.exists():
        cpi = pd.read_csv(INFLATION_CACHE, parse_dates=["DATE"], index_col="DATE")
        return cpi.loc["2021-01-01":"2026-03-11"]

    # FRED blocks urllib's default agent — use requests with a browser header
    response = requests.get(
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL",
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    response.raise_for_status()
    cpi = pd.read_csv(io.StringIO(response.text))
    cpi.columns = ["DATE", "CPIAUCSL"]
    cpi["DATE"] = pd.to_datetime(cpi["DATE"])
    cpi = cpi.set_index("DATE")
    cpi["YoY_Inflation"] = cpi["CPIAUCSL"].pct_change(periods=12) * 100
    cpi = cpi.dropna()
    cpi.to_csv(INFLATION_CACHE)
    return cpi.loc["2021-01-01":"2026-03-11"]


def plot_close_prices(df):
    yoy_inflation = get_inflation()

    fig = px.line(
        df,
        x="Date",
        y="Close",
        color="Company",
        title="Stock Closing Prices Over Time",
        labels={"Close": "Closing Price (USD)"},
    )

    inflation_trace = go.Scatter(
        x=yoy_inflation.index,
        y=yoy_inflation["YoY_Inflation"],
        name="US Inflation Rate (YoY %)",
        fill="tozeroy",
        mode="lines",
        line=dict(color="rgba(64, 64, 64, 0.9)", width=4, dash="dot"),
        fillcolor="rgba(128, 128, 128, 0.25)",
        yaxis="y2"
    )
    fig.add_trace(inflation_trace)

    # Push inflation trace to the back
    fig.data = (fig.data[-1],) + fig.data[:-1]

    fig.update_layout(
        height=800,
        width=1400,
        template="plotly_white",
        yaxis2=dict(
            title="YoY Inflation Rate (%)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        legend=dict(
            title="<b>Assets</b>",
            itemsizing="constant"
        )
    )

    fig.show()


plot_close_prices(df)


KeyboardInterrupt



### Correlation Analysis: Cross-Asset Correlation During Stress Periods

Assets that appear uncorrelated under normal conditions often converge during market stress — exactly when diversification is needed most. This section compares correlation structure across normal and stressed market regimes to identify which assets hold their diversification properties under pressure.

In [18]:
# Pivot to one column per ticker, then compute weekly percentage returns
# We correlate returns, not prices — prices are non-stationary and give misleading correlations
pivot = df.pivot_table(index="Date", columns="Ticker", values="Close")
returns = pivot.pct_change().dropna()
tickers = returns.columns.tolist()

print(f"{len(tickers)} tickers, {len(returns)} weekly observations")
returns.tail()

27 tickers, 270 weekly observations


Ticker,AAPL,ABT,AGG,AMD,AMZN,BITW,BND,DIA,HD,ITA,...,SPY,TLT,TSLA,VNQ,VTI,WMT,XAR,XLE,XLP,XLU
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-06,-0.051394,0.021911,0.009272,0.069818,-0.103687,0.033874,0.009259,0.011781,0.020530,0.028075,...,0.005387,0.023906,0.049999,0.026756,0.006319,0.052781,0.031999,0.033902,0.026346,0.049884
2026-02-13,-0.003462,0.006459,0.002086,-0.012479,0.026353,0.025743,0.002143,-0.000586,-0.029829,0.047134,...,0.004712,0.004371,-0.012852,0.009866,0.006488,-0.065624,0.062687,0.022230,-0.017263,0.019006
2026-02-20,0.047471,0.036278,0.003172,0.001524,0.014937,0.013689,0.002673,0.000971,-0.009219,-0.002752,...,0.007042,0.007253,-0.007602,0.014336,0.006801,-0.003604,-0.002637,-0.002356,0.013574,0.023205
2026-02-27,-0.046382,-0.044899,-0.009486,-0.020768,0.053001,0.047265,-0.009064,-0.029148,-0.035751,-0.011449,...,-0.011591,-0.016395,-0.007416,-0.009841,-0.013187,-0.008921,-0.017412,0.025976,-0.038825,-0.005935
2026-03-06,0.001998,-0.007115,-0.002678,0.026974,-0.028729,-0.006877,-0.003029,-0.010483,-0.029971,-0.005666,...,-0.007309,-0.015320,0.005597,-0.020406,-0.008214,0.001460,-0.004735,0.008853,-0.009601,-0.015565


In [19]:
# Define major stress periods within the 2021-2026 window
stress_periods = {
    "2022 Bear Market":    ("2022-01-01", "2022-10-31"),  # Fed rate hikes, SPY -25%
    "2023 Banking Crisis": ("2023-03-01", "2023-05-31"),  # SVB collapse
    "2025 Tariff Shock":   ("2025-02-01", "2025-04-30"),  # Trump tariff escalation
}

# All observations that fall outside every stress window
normal_mask = pd.Series(True, index=returns.index)
for start, end in stress_periods.values():
    normal_mask &= ~((returns.index >= start) & (returns.index <= end))

normal_returns = returns[normal_mask]
stress_2022    = returns["2022-01-01":"2022-10-31"]

print(f"Normal observations:      {len(normal_returns)}")
print(f"2022 stress observations: {len(stress_2022)}")

Normal observations:      202
2022 stress observations: 43


In [20]:
# numpy.corrcoef: correlation matrices per market regime
# corrcoef expects variables as rows — transpose so each row is one ticker's return series
corr_normal = np.corrcoef(normal_returns.T)
corr_stress  = np.corrcoef(stress_2022.T)
corr_full    = np.corrcoef(returns.T)

In [21]:
# scipy.stats.pearsonr: attach p-values to every pair
# A correlation of 0.4 on 40 weekly observations may not be statistically meaningful
n = len(tickers)
pval_matrix = np.ones((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            _, pval_matrix[i, j] = stats.pearsonr(
                returns.iloc[:, i], returns.iloc[:, j]
            )

sig_mask = pval_matrix < 0.05
corr_significant = np.where(sig_mask, corr_full, np.nan)

sig_count   = int(sig_mask.sum()) - n   # subtract self-correlations on diagonal
total_pairs = n * (n - 1)
print(f"{sig_count}/{total_pairs} pairs statistically significant (p < 0.05)")

579/702 pairs statistically significant (p < 0.05)


In [22]:
# scipy.stats.spearmanr: rank-based correlation — more robust when returns have fat tails
spearman_corr, spearman_pval = stats.spearmanr(returns)
spearman_df = pd.DataFrame(spearman_corr, index=tickers, columns=tickers)

#### Normal vs. Stress Heatmaps

In [23]:
def corr_heatmap(matrix, labels, title):
    fig = px.imshow(
        pd.DataFrame(matrix, index=labels, columns=labels),
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title=title,
    )
    fig.update_layout(width=900, height=800, template="plotly_white")
    return fig


corr_heatmap(corr_normal, tickers, "Pearson Correlations — Normal Periods").show()
corr_heatmap(corr_stress, tickers, "Pearson Correlations — 2022 Bear Market").show()

#### Correlation Shift: Where Did Diversification Break Down?

Subtracting normal-period correlations from stress-period correlations reveals which pairs converged (failed to diversify) and which diverged (held up as true diversifiers).

- **Red (positive delta):** assets became MORE correlated under stress — diversification failed  
- **Blue (negative delta):** assets diverged under stress — genuine diversification benefit

In [24]:
corr_normal_df = pd.DataFrame(corr_normal, index=tickers, columns=tickers)
corr_stress_df = pd.DataFrame(corr_stress, index=tickers, columns=tickers)
delta = corr_stress_df - corr_normal_df

corr_heatmap(
    delta.values,
    tickers,
    "Correlation Shift: 2022 Stress vs. Normal (red = converged, blue = diverged)"
).show()

#### Spearman Rank Correlation — Full Period

Spearman is more robust than Pearson when distributions have fat tails, which asset returns consistently do. Use this as a sanity check against the Pearson results above.

In [25]:
corr_heatmap(
    spearman_df.values,
    tickers,
    "Spearman Rank Correlation — Full Period (Robust to Fat Tails)"
).show()

#### Statistically Significant Correlations Only (p < 0.05)

Blank cells indicate the correlation exists but is not statistically significant — treat those pairs as effectively uncorrelated.

In [26]:
corr_heatmap(
    corr_significant,
    tickers,
    "Significant Correlations Only (p < 0.05)"
).show()

XLE has the most independent/divergent outcomes, with even further negative correlation with TLT, iShares 20+ Year Treasury Bond ETF.